# TabTransformer — Attention over Categorical Features
> **Research question:** Does applying transformer attention to categorical
> feature embeddings capture cross-feature interactions that tree models miss?

**Architecture summary**
- Each categorical column → learned embedding vector (d=64)
- Each numeric feature → its own projected token (one token per feature)
- All tokens → 3-layer TransformerEncoder (4 attention heads)
- Flattened sequence → MLP head → fraud probability

**Why this might help:** Attention learns *which pairs of features* matter for a
given transaction (e.g., card type × email domain), rather than treating features
independently.

In [ ]:
# Install required packages (run once in Colab)
!pip install -q torch pandas numpy scikit-learn pyarrow lightgbm

## 1 · Data Configuration

In [ ]:
import os

# ── Kaggle API download (default — works out of the box in Colab) ─────────────
# Prerequisites (one-time setup):
#   1. Go to kaggle.com → Account → API → "Create New Token" → downloads kaggle.json
#   2. In Colab: Secrets (🔑 icon in left sidebar) → add KAGGLE_USERNAME and KAGGLE_KEY
#      (copy the values from your kaggle.json file)
#   3. Run this cell — it will download and unzip the data automatically

from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

DATA_DIR = "data"
os.makedirs(f"{DATA_DIR}/raw",     exist_ok=True)
os.makedirs(f"{DATA_DIR}/interim", exist_ok=True)

print("Downloading IEEE-CIS Fraud Detection dataset from Kaggle …")
os.system("pip install -q kaggle")
os.system(f"kaggle competitions download -c ieee-fraud-detection -p {DATA_DIR}/raw/ --unzip -q")
print("Download complete. Files in data/raw/:")
os.system(f"ls -lh {DATA_DIR}/raw/")

## 2 · Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

# CUDA on Colab T4/A100, MPS on Apple Silicon, CPU fallback
def get_device():
    if torch.cuda.is_available():         return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Device: {device}")

## 3 · Load & Merge Data
Left-joins the two source files so every transaction is represented.
Identity features (~76% missing) are NaN where unavailable.

In [ ]:
import os
import pandas as pd

def load_merged_train(data_dir="data"):
    """
    Load and merge the IEEE-CIS training files.

    Reads train_transaction.csv and train_identity.csv from data_dir/raw/,
    left-joins them on TransactionID so every transaction is kept (identity
    features are NaN for ~76% of rows that have no identity record), then
    caches the result to data_dir/interim/merged_train.parquet for fast
    re-loads.

    Returns
    -------
    pd.DataFrame  shape ~ (590540, 434)
    """
    cache = os.path.join(data_dir, "interim", "merged_train.parquet")
    if os.path.exists(cache):
        print(f"Loading from cache: {cache}")
        df = pd.read_parquet(cache)
        print(f"Shape: {df.shape}")
        return df

    raw = os.path.join(data_dir, "raw")
    print("Reading train_transaction.csv …")
    trans = pd.read_csv(os.path.join(raw, "train_transaction.csv"))
    print("Reading train_identity.csv …")
    ident = pd.read_csv(os.path.join(raw, "train_identity.csv"))
    print("Merging on TransactionID …")
    df = trans.merge(ident, how="left", on="TransactionID")
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    df.to_parquet(cache, index=False)
    print(f"Cached to {cache}.  Shape: {df.shape}")
    return df

df = load_merged_train(DATA_DIR)
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(f"Sorted by TransactionDT.  Shape: {df.shape}")

## 4 · Temporal Train / Val / Test Split
**70 / 15 / 15** split respecting `TransactionDT` order.
Split is defined *before* feature selection so that LightGBM importance
is computed on training data only — no leakage.

In [ ]:
# Temporal 70 / 15 / 15 split — respects transaction time ordering.
# Using random splits on time-series data leaks future info into training.
n        = len(df)
n_train  = int(0.70 * n)
n_val    = int(0.15 * n)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_train + n_val))
test_idx  = list(range(n_train + n_val, n))
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")

## 5 · Feature Selection via LightGBM Importance
Instead of heuristically taking the first 50 numeric columns (which would give
V1–V11, not necessarily the best V-features), we run a quick 100-tree LightGBM
and rank every candidate feature by how often it is used in splits.
Both models use the same features for a fair comparison.

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

def select_features_by_importance(df, train_idx, target_col="isFraud",
                                   max_numeric=50, max_categorical=20):
    """
    Rank all candidate features by LightGBM split importance, then pick the top N.

    Why this beats heuristic selection:
      The IEEE-CIS dataset has 339 anonymised V-columns — their names reveal
      nothing about their value. Heuristic selection (first 50 columns) grabs
      V1-V11, which may be less informative than V45 or V258. LightGBM quickly
      identifies which features the trees actually use for splits.

    Process:
      1. Drop columns that are >95% missing.
      2. Classify remaining columns as numeric or categorical.
      3. Fit a fast LightGBM (100 trees) on the TRAINING split only.
      4. Rank features by cumulative split gain.
      5. Return top max_numeric numeric + top max_categorical categorical.
    """
    ignore = {target_col, "TransactionID"}
    na_frac = df.isna().mean()
    kept = [c for c in na_frac[na_frac < 0.95].index if c not in ignore]

    num_cands, cat_cands = [], []
    for col in kept:
        dt = df[col].dtype
        if dt == "object" or str(dt) == "category":
            cat_cands.append(col)
        elif np.issubdtype(dt, np.integer):
            (cat_cands if df[col].nunique(dropna=True) <= 20 else num_cands).append(col)
        elif np.issubdtype(dt, np.floating):
            num_cands.append(col)

    all_cands = num_cands + cat_cands
    X = df[all_cands].copy()
    for col in cat_cands:
        X[col] = X[col].astype("category").cat.codes  # -1 for NaN, fine for trees
    X = X.fillna(-999).values.astype(np.float32)
    y = df[target_col].values

    pos = (y[train_idx] == 1).sum()
    neg = (y[train_idx] == 0).sum()
    print("Running LightGBM for feature importance (100 trees) …")
    lgb = LGBMClassifier(n_estimators=100, n_jobs=-1, verbose=-1,
                          random_state=42, scale_pos_weight=neg/pos)
    lgb.fit(X[train_idx], y[train_idx])

    importance = pd.Series(lgb.feature_importances_, index=all_cands).sort_values(ascending=False)
    num_set, cat_set = set(num_cands), set(cat_cands)
    numeric_cols     = [c for c in importance.index if c in num_set][:max_numeric]
    categorical_cols = [c for c in importance.index if c in cat_set][:max_categorical]

    print(f"Selected {len(numeric_cols)} numeric + {len(categorical_cols)} categorical by importance")
    print("Top 10 numeric:   ", numeric_cols[:10])
    print("Categorical:      ", categorical_cols)
    return numeric_cols, categorical_cols

numeric_cols, categorical_cols = select_features_by_importance(
    df, train_idx, target_col="isFraud", max_numeric=50, max_categorical=20
)

## 6 · Model Architecture — CustomTabTransformer

Each input feature becomes a fixed-size token:
- **Categoricals** → learned embedding matrix (vocab_size × d_token)
- **Numerics** → scalar multiplied by a learned weight vector, plus a bias vector
  (one weight/bias pair per numeric feature — equivalent to a linear projection per feature)

All tokens are then passed through a standard TransformerEncoder.
The output tokens are flattened and fed to an MLP that produces a single fraud logit.

In [ ]:
class CustomTabTransformer(nn.Module):
    """
    TabTransformer-style model built from scratch.

    Parameters
    ----------
    vocab_sizes : list[int]
        Number of unique values for each categorical column.
    num_numeric_features : int
        Number of numeric input columns.
    d_token : int
        Embedding / token dimension (default 64).
    n_heads : int
        Number of attention heads (default 4). Must divide d_token evenly.
    n_layers : int
        Number of TransformerEncoder layers (default 3).
    dropout : float
        Dropout probability applied inside transformer and MLP head (default 0.1).
    """

    def __init__(self, vocab_sizes, num_numeric_features,
                 d_token=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.d_token = d_token
        self.n_tokens = num_numeric_features + len(vocab_sizes)

        # One embedding table per categorical column
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(n_cat, d_token) for n_cat in vocab_sizes
        ])

        # Per-numeric-feature linear projection: x_i → x_i * w_i + b_i
        self.num_weight = nn.Parameter(torch.randn(num_numeric_features, d_token) * 0.02)
        self.num_bias   = nn.Parameter(torch.zeros(num_numeric_features, d_token))

        # Transformer encoder (batch_first so input shape is B × T × D)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=4 * d_token, dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # Classification head: LayerNorm → Linear → ReLU → Dropout → Linear → logit
        self.head = nn.Sequential(
            nn.LayerNorm(self.n_tokens * d_token),
            nn.Linear(self.n_tokens * d_token, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x_num, x_cat):
        # Numeric tokens: (B, n_num) → (B, n_num, d_token)
        num_tokens = x_num.unsqueeze(2) * self.num_weight + self.num_bias
        # Categorical tokens: (B, n_cat) → (B, n_cat, d_token)
        cat_tokens = torch.stack(
            [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)], dim=1
        )
        tokens  = torch.cat([num_tokens, cat_tokens], dim=1)   # (B, T, d_token)
        encoded = self.transformer(tokens)
        return self.head(encoded.reshape(encoded.size(0), -1)).squeeze(-1)

## 7 · Training Function

Key training decisions:
- **Mini-batch (512)** — full dataset is 590k rows; mini-batching is required
- **BCEWithLogitsLoss with pos_weight** — compensates for 3.5% fraud rate
- **AdamW + ReduceLROnPlateau** — LR halves only when val PR-AUC stops improving
  (avoids the aggressive decay of CosineAnnealingLR which hurt performance)
- **Early stopping (patience=5)** — saves the best checkpoint and stops if val
  PR-AUC doesn't improve for 5 consecutive epochs
- **Threshold tuning** — sweeps the PR curve to find the F1-maximising threshold

In [ ]:
def train_tabtransformer(df, numeric_cols, categorical_cols,
                         target_col="isFraud", batch_size=2048, num_epochs=50,
                         train_idx=None, val_idx=None, test_idx=None,
                         early_stopping_patience=10, warmup_epochs=3):
    """
    Train the CustomTabTransformer on the given DataFrame.

    Returns
    -------
    metrics : dict   val + test precision / recall / F1 / ROC-AUC / PR-AUC
    model   : nn.Module   best checkpoint (highest val PR-AUC)
    """
    device = get_device()
    print(f"Using device: {device}")

    # ── Vocab sizes + categorical encoding ───────────────────────────────────
    vocab_sizes, cat_mappings = [], []
    for col in categorical_cols:
        vals    = df[col].astype(str)
        uniques = list(vals.unique())
        vocab_sizes.append(len(uniques))
        cat_mappings.append({v: i for i, v in enumerate(uniques)})

    # ── Numeric tensor (StandardScaler fit on train only) ────────────────────
    num_data = df[numeric_cols].fillna(0).values.astype(np.float32)
    if train_idx is not None:
        scaler = StandardScaler()
        num_data[train_idx] = scaler.fit_transform(num_data[train_idx])
        if val_idx  is not None: num_data[val_idx]  = scaler.transform(num_data[val_idx])
        if test_idx is not None: num_data[test_idx] = scaler.transform(num_data[test_idx])
    x_num = torch.tensor(num_data, dtype=torch.float32)

    # ── Categorical tensor ────────────────────────────────────────────────────
    x_cat = torch.zeros((len(df), len(categorical_cols)), dtype=torch.long)
    for i, col in enumerate(categorical_cols):
        x_cat[:, i] = torch.tensor(
            [cat_mappings[i][v] for v in df[col].astype(str)], dtype=torch.long
        )

    y = torch.tensor(df[target_col].values, dtype=torch.float32)

    # ── Splits ────────────────────────────────────────────────────────────────
    if train_idx is None or val_idx is None:
        n = len(df); n_tr = int(0.8 * n)
        train_idx = list(range(0, n_tr)); val_idx = list(range(n_tr, n))

    y_train = y[train_idx]
    pos_weight = ((y_train == 0).sum().float() /
                  (y_train == 1).sum().float()).to(device)
    print(f"Class imbalance → pos_weight = {pos_weight.item():.2f}")

    # ── DataLoaders ───────────────────────────────────────────────────────────
    def make_loader(idx, shuffle):
        ds = TensorDataset(x_num[idx], x_cat[idx], y[idx])
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2)

    train_loader = make_loader(train_idx, shuffle=True)
    val_loader   = make_loader(val_idx,   shuffle=False)

    # ── Model / optimizer / schedulers ───────────────────────────────────────
    model     = CustomTabTransformer(vocab_sizes, len(numeric_cols)).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    # LR scaled linearly with batch size (batch 2048 vs reference 512 → 4× → 2e-3 → 4e-3... use 2e-3)
    optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    # Linear warmup for first `warmup_epochs` — prevents bad early gradients
    # from corrupting embeddings before training stabilises
    warmup_sched = optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda ep: (ep + 1) / warmup_epochs if ep < warmup_epochs else 1.0
    )
    # After warmup: halve LR only when val PR-AUC stalls for 3 epochs
    plateau_sched = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    best_pr_auc, best_epoch, patience_ctr, best_state = -1.0, 0, 0, None

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for bx_num, bx_cat, by in train_loader:
            bx_num, bx_cat, by = bx_num.to(device), bx_cat.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx_num, bx_cat), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item(); n_batches += 1
            del bx_num, bx_cat, by, loss
            if device.type == "mps" and n_batches % 100 == 0:
                torch.mps.empty_cache()

        if device.type == "mps": torch.mps.empty_cache()

        # Validation PR-AUC
        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for bx_num, bx_cat, by in val_loader:
                val_logits.append(model(bx_num.to(device), bx_cat.to(device)).cpu())
                val_labels.append(by)
        ep_probs    = torch.sigmoid(torch.cat(val_logits)).numpy()
        ep_true     = torch.cat(val_labels).numpy()
        ep_pr_auc   = average_precision_score(ep_true, ep_probs)
        avg_loss    = total_loss / max(n_batches, 1)

        # Warmup for first N epochs, then adaptive LR
        if epoch <= warmup_epochs:
            warmup_sched.step()
        else:
            plateau_sched.step(ep_pr_auc)
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"[Epoch {epoch:3d}] loss={avg_loss:.4f}  val_pr_auc={ep_pr_auc:.4f}  lr={current_lr:.2e}")

        if ep_pr_auc > best_pr_auc:
            best_pr_auc, best_epoch, patience_ctr = ep_pr_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch}. Best: epoch {best_epoch} (pr_auc={best_pr_auc:.4f})")
                break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"Restored best model from epoch {best_epoch}")

    # ── Final evaluation ──────────────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for bx_num, bx_cat, by in val_loader:
            all_logits.append(model(bx_num.to(device), bx_cat.to(device)).cpu())
            all_labels.append(by)
    y_score = torch.sigmoid(torch.cat(all_logits)).numpy()
    y_true  = torch.cat(all_labels).numpy()

    precs, recs, thrs = precision_recall_curve(y_true, y_score)
    f1s = 2 * precs[:-1] * recs[:-1] / (precs[:-1] + recs[:-1] + 1e-8)
    best_thr = float(thrs[f1s.argmax()])
    y_pred   = (y_score >= best_thr).astype(float)

    metrics = {
        "val_precision": precision_score(y_true, y_pred, zero_division=0),
        "val_recall":    recall_score(y_true, y_pred, zero_division=0),
        "val_f1":        f1_score(y_true, y_pred, zero_division=0),
        "val_roc_auc":   roc_auc_score(y_true, y_score),
        "val_pr_auc":    average_precision_score(y_true, y_score),
        "val_threshold": best_thr,
    }
    print("\n===== TabTransformer — Validation =====")
    for k, v in metrics.items(): print(f"  {k:20s}: {v:.4f}")

    # Test set
    if test_idx is not None:
        test_loader = DataLoader(
            TensorDataset(x_num[test_idx], x_cat[test_idx], y[test_idx]),
            batch_size=batch_size, shuffle=False, num_workers=2,
        )
        test_logits = []
        with torch.no_grad():
            for bx_num, bx_cat, _ in test_loader:
                test_logits.append(model(bx_num.to(device), bx_cat.to(device)).cpu())
        y_test_score = torch.sigmoid(torch.cat(test_logits)).numpy()
        y_test_true  = y[test_idx].numpy()
        y_test_pred  = (y_test_score >= best_thr).astype(float)
        test_m = {
            "test_precision": precision_score(y_test_true, y_test_pred, zero_division=0),
            "test_recall":    recall_score(y_test_true, y_test_pred, zero_division=0),
            "test_f1":        f1_score(y_test_true, y_test_pred, zero_division=0),
            "test_roc_auc":   roc_auc_score(y_test_true, y_test_score),
            "test_pr_auc":    average_precision_score(y_test_true, y_test_score),
        }
        print("\n===== TabTransformer — Test =====")
        for k, v in test_m.items(): print(f"  {k:20s}: {v:.4f}")
        metrics.update(test_m)

    return metrics, model

## 8 · Run Training

In [ ]:
metrics, model = train_tabtransformer(
    df, numeric_cols, categorical_cols,
    target_col="isFraud",
    batch_size=2048,          # large batch — T4/A100 GPU has plenty of VRAM
    num_epochs=50,            # more room with adaptive LR + patience=10
    early_stopping_patience=10,
    warmup_epochs=3,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
)

## 9 · Results Summary

In [ ]:
print("\n" + "="*45)
print("  TabTransformer — Final Results")
print("="*45)
for k, v in metrics.items():
    print(f"  {k:22s}: {v:.4f}")